In [1]:
import random
import time
class Thing:    
    """Represents a physical object in the environment."""
    def is_alive(self):
        return hasattr(self, "alive") and self.alive    
    def show_state(self):       
        print("I don't know how to show_state.")

In [2]:
class Agent(Thing):    
    """An agent that selects actions using an agent program."""   
    def __init__(self, program=None):        
        self.alive = True        
        self.performance = 0        
        self.program = program   
    def can_grab(self, thing):        
        return False

In [3]:
def TableDrivenAgentProgram(table):
    """Select an action using the current percept and a lookup table."""
    def program(percept):
        return table.get(percept, "NoOp")
    return program

In [4]:
# Two hospital room locations
room_A, room_B = (0, 0), (1, 0)

In [5]:
def TableDrivenDoctorAgent():
    """Create a table-driven doctor agent for the two-room environment."""
    table = {
        (room_A, "unhealthy"): "treat",
        (room_A, "healthy"): "Right",
        (room_B, "unhealthy"): "treat",
        (room_B, "healthy"): "Left"
    }
    return Agent(TableDrivenAgentProgram(table))

In [6]:
TableDrivenDoctorAgent()

In [7]:
class Environment:
    """Base class for an agent environment."""
    def __init__(self):
        self.things = []
        self.agents = []
        def percept(self, agent):
            raise NotImplementedError
            def execute_action(self, agent, action):
                raise NotImplementedError
                def default_location(self, thing):        
                    return None
                def is_done(self):
                    return not any(agent.is_alive() for agent in self.agents)
                def step(self):
                    if self.is_done():
                        return
                    for agent in self.agents:
                        if agent.alive:
                            action = agent.program(self.percept(agent))
                            self.execute_action(agent, action)
                            def run(self, steps=100):
                                for _ in range(steps):
                                    if self.is_done():
                                        break
                                        self.step()
                                        def add_thing(self, thing, location=None):
                                            if not isinstance(thing, Thing):
                                                thing = Agent(thing)
                                                if thing in self.things:
                                                    print("Can't add the same thing twice.")
                                                    return
                                                thing.location = location if location is not None else self.default_location(thing)
                                                self.things.append(thing)
                                                if isinstance(thing, Agent):
                                                    thing.performance = 0
                                                    self.agents.append(thing)
                                                    def delete_thing(self, thing):
                                                        if thing in self.things:
                                                            self.things.remove(thing)
                                                            if thing in self.agents:
                                                                self.agents.remove(thing)
                                                                

In [8]:
class Environment:
    """Abstract class representing an Environment. 'Real' Environment classes inherit from this. Your Environment will typically need to implement:
    percept:	Define the percept that an agent sees. execute_action:	Define the effects of executing an action.
    Also update the agent.performance slot.
    The environment keeps a list of .things and .agents (which is a subset of .things). Each agent has a .performance slot, initialized to 0.
    Each thing has a .location slot, even though some environments may not need this."""

    def __init__(self):
        self.things = [] 
        self.agents = []
        #room_A, room_B = (0,0), (1,0) # The two locations for the Doctor to treat

    def percept(self, agent):
        """Return the percept that the agent sees at this point. (Implement this.)"""
        raise NotImplementedError

    def execute_action(self, agent, action):
        """Change the world to reflect this action. (Implement this.)""" 
        raise NotImplementedError

    def default_location(self, thing):
        """Default location to place a new thing with unspecified location."""
        return None

    def is_done(self):
        """By default, we're done when we can't find a live agent.""" 
        return not any(agent.is_alive() for agent in self.agents)

    def step(self):
        """Run the environment for one time step. If the
        actions and exogenous changes are independent, this method will do. If there are interactions between them, you'll need to override this method."""
        if not self.is_done(): 
            actions = []
            for agent in self.agents:
                if agent.alive:
                    actions.append(agent.program(self.percept(agent))) 
                else:
                    actions.append("")
            for (agent, action) in zip(self.agents, actions): 
                self.execute_action(agent, action)

    def run(self, steps=1000):
        """Run the Environment for given number of time steps."""
        for step in range(steps):
            if self.is_done():
                return 
            self.step()

    def add_thing(self, thing, location=None):
        """Add a thing to the environment, setting its location. For convenience, if thing is an agent program we make a new agent for it. (Shouldn't need to override this.)"""
        if not isinstance(thing, Thing):
            thing = Agent(thing)
        if thing in self.things:
            print("Can't add the same thing twice") 
        else:
            thing.location = (location if location is not None else self.default_location(thing))
            self.things.append(thing) 
            if isinstance(thing, Agent):
                thing.performance = 0 
                self.agents.append(thing)

    def delete_thing(self, thing):
        """Remove a thing from the environment."""
        try:
            
            self.things.remove(thing) 
        except ValueError as e:
            print(e)
            print(" in Environment delete_thing")
            print(" Thing to be removed: {} at {}".format(thing, thing.location))
            print(" from list: {}".format([(thing, thing.location) for thing in self.things]))
        if thing in self.agents: 
            self.agents.remove(thing)

In [9]:

class TrivialDoctorEnvironment(Environment):
    """Two-room hospital. Patient status is based on entered temperature."""

    def __init__(self):
        super().__init__()

        print("Enter patient temperatures in Fahrenheit.")
        temp_A = float(input("Room A temperature: "))
        temp_B = float(input("Room B temperature: "))

        # Temperature and patient status
        self.temperatures = {
            room_A: temp_A,
            room_B: temp_B
        }

        self.status = {
            room_A: "unhealthy" if temp_A > 98.5 else "healthy",
            room_B: "unhealthy" if temp_B > 98.5 else "healthy"
        }

        self.visited_rooms = set()

    def percept(self, agent):
        self.visited_rooms.add(agent.location)
        return agent.location, self.status[agent.location]

    def execute_action(self, agent, action):
        if action == "Right":
            agent.location = room_B
            agent.performance -= 1
            print("Agent moves to Room B. Performance:", agent.performance)

        elif action == "Left":
            agent.location = room_A
            agent.performance -= 1
            print("Agent moves to Room A. Performance:", agent.performance)

        elif action == "treat":
            print(
                "Unhealthy patient detected in",
                "Room A" if agent.location == room_A else "Room B"
            )
            print("Action: Simulated treatment completed.")

            self.status[agent.location] = "healthy"
            agent.performance += 10
            print("Performance:", agent.performance)

        elif action == "NoOp":
            agent.alive = False
            print("No action available. Agent stopped.")

    def default_location(self, thing):
        return random.choice([room_A, room_B])

    def is_done(self):
        # Finish after both rooms have been visited and are healthy.
        return (
            len(self.visited_rooms) == 2
            and all(state == "healthy" for state in self.status.values())
        )


In [10]:
# Run the Medicine Prescribing / Doctor Agent simulation
if __name__ == "__main__":
    agent = TableDrivenDoctorAgent()
    environment = TrivialDoctorEnvironment()
    environment.add_thing(agent)
    print("\nPatient status before treatment:")
    print(environment.status)
    print("Agent starting location:", agent.location)
    print("Initial performance:", agent.performance)
    environment.run(steps=10)
    print("\nPatient status after the simulation:")
    print(environment.status)
    print("Final agent location:", agent.location)    
    print("Final performance:", agent.performance)
    print("All rooms checked:", all(
        state == "healthy" for state in environment.status.values()
    ))

Enter patient temperatures in Fahrenheit.
Room A temperature: 101
Room B temperature: 98

Patient status before treatment:
{(0, 0): 'unhealthy', (1, 0): 'healthy'}
Agent starting location: (1, 0)
Initial performance: 0
Agent moves to Room A. Performance: -1
Unhealthy patient detected in Room A
Action: Simulated treatment completed.
Performance: 9

Patient status after the simulation:
{(0, 0): 'healthy', (1, 0): 'healthy'}
Final agent location: (0, 0)
Final performance: 9
All rooms checked: True
